# AD9226 Capture Test — 31.25MHz PL Clock Version

这版 notebook 按你现在实测结果设置：

```text
PL_CLK_HZ = 31_250_000
Fs_adc = PL_CLK_HZ / (2 * adc_half_period)
```

前面 `mode 0` 和 `mode 2` 你已经测过，可以直接跳到 **真实 AD9226 测试**。  
但如果重新上传了 bit/hwh，建议先快速跑一次 **Quick Mode 0/2**。

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import matplotlib.pyplot as plt
from time import time, sleep

overlay = Overlay("base_add.bit")

print("IP list:")
for k in overlay.ip_dict.keys():
    print("  ", k)

writer = overlay.base_add_0
ctrl = overlay.adc_capture_0

# led_ctrl_0 可选，不一定用
led = overlay.led_ctrl_0 if "led_ctrl_0" in overlay.ip_dict else None

print("\nwriter register_map:")
print(writer.register_map)

print("\nadc_capture_0 info:")
print(overlay.ip_dict["adc_capture_0"])

print("\nLoad OK")

## 1. 寄存器、参数、公共函数

只需要运行一次。  
这里已经把 `PL_CLK_HZ` 设置为 **31.25MHz**。

In [ ]:
# ===== adc_capture_0 registers =====
CTRL          = 0x00
STATUS        = 0x04
SAMPLE_COUNT  = 0x08
ADC_HALF      = 0x0C
SAMPLE_DELAY  = 0x10
DECIMATION    = 0x14
CHANNEL_MASK  = 0x18
CAPTURE_MODE  = 0x1C
TRIGGER_MODE  = 0x20
PRE_DELAY     = 0x24
BUFFER_SELECT = 0x28
LATEST_A      = 0x2C
LATEST_B      = 0x30
SAMPLE_CNTR   = 0x34
FIFO_LEVEL    = 0x38
ERROR_FLAGS   = 0x3C
LED_CTRL      = 0x40
VERSION       = 0x44
SAVED_COUNTER = 0x48
LAST_SAMPLE   = 0x4C
DEBUG_STATE   = 0x50

# ===== HLS writer registers =====
WRITER_AP_CTRL      = 0x00
WRITER_BUFFER       = 0x10
WRITER_COUNT        = 0x18
WRITER_CAPTURE_MODE = 0x20

# ===== constants =====
MAX_SAMPLE_N = 65536
BUFFER_WORDS = 2 * MAX_SAMPLE_N

# 你当前逻辑分析仪实测 PL FCLK≈31.25MHz
PL_CLK_HZ = 31_250_000

def decode_status(status):
    return {
        "busy":          bool(status & (1 << 0)),
        "done":          bool(status & (1 << 1)),
        "adc_clk_seen":  bool(status & (1 << 2)),
        "fifo_full":     bool(status & (1 << 3)),
        "fifo_empty":    bool(status & (1 << 4)),
        "fifo_overflow": bool(status & (1 << 5)),
        "near_rail_a":   bool(status & (1 << 6)),
        "near_rail_b":   bool(status & (1 << 7)),
        "writer_busy":   bool(status & (1 << 8)),
        "writer_done":   bool(status & (1 << 9)),
        "error":         bool(status & (1 << 10)),
        "data_changed_a":bool(status & (1 << 11)),
        "data_changed_b":bool(status & (1 << 12)),
    }

def read_status(verbose=True):
    status = ctrl.read(STATUS)
    err = ctrl.read(ERROR_FLAGS)
    fifo = ctrl.read(FIFO_LEVEL)
    saved = ctrl.read(SAVED_COUNTER)
    last = ctrl.read(LAST_SAMPLE)
    dbg = ctrl.read(DEBUG_STATE)
    writer_ctrl = writer.read(WRITER_AP_CTRL)
    la = ctrl.read(LATEST_A)
    lb = ctrl.read(LATEST_B)
    sc = ctrl.read(SAMPLE_CNTR)
    ver = ctrl.read(VERSION)

    if verbose:
        print("STATUS       = 0x%08X" % status, decode_status(status))
        print("ERROR_FLAGS  = 0x%08X" % err)
        print("FIFO_LEVEL   =", fifo)
        print("SAVED_CNTR   =", saved)
        print("LAST_SAMPLE  = 0x%08X" % last)
        print("DEBUG_STATE  =", dbg)
        print("WRITER_CTRL  = 0x%08X" % writer_ctrl)
        print("LATEST_A     =", la)
        print("LATEST_B     =", lb)
        print("SAMPLE_CNTR  =", sc)
        print("VERSION      = 0x%08X" % ver)

    return {
        "status": status,
        "decoded": decode_status(status),
        "error_flags": err,
        "fifo_level": fifo,
        "saved_counter": saved,
        "last_sample": last,
        "debug_state": dbg,
        "writer_ctrl": writer_ctrl,
        "latest_a": la,
        "latest_b": lb,
        "sample_counter": sc,
        "version": ver,
    }

def clear_ctrl():
    # 清错误 + clear pulse
    ctrl.write(ERROR_FLAGS, 0xFFFFFFFF)
    ctrl.write(CTRL, 0x04)
    ctrl.write(CTRL, 0x00)

def configure_ctrl(sample_count=1024,
                   adc_half_period=25,
                   sample_delay=1,
                   decimation=1,
                   channel_mask=0b11,
                   capture_mode=1,
                   trigger_mode=0,
                   pre_delay=0,
                   buffer_select=0,
                   led_ps_override=0,
                   led_value=0):

    sample_count = min(max(int(sample_count), 1), MAX_SAMPLE_N)
    adc_half_period = max(int(adc_half_period), 1)
    decimation = max(int(decimation), 1)

    clear_ctrl()

    ctrl.write(SAMPLE_COUNT, sample_count)
    ctrl.write(ADC_HALF, adc_half_period)
    ctrl.write(SAMPLE_DELAY, sample_delay)
    ctrl.write(DECIMATION, decimation)
    ctrl.write(CHANNEL_MASK, channel_mask)
    ctrl.write(CAPTURE_MODE, capture_mode)
    ctrl.write(TRIGGER_MODE, trigger_mode)
    ctrl.write(PRE_DELAY, pre_delay)
    ctrl.write(BUFFER_SELECT, buffer_select)
    ctrl.write(LED_CTRL, (led_ps_override << 8) | (led_value & 0xF))

    # writer 也必须同步写 capture_mode
    writer.write(WRITER_CAPTURE_MODE, capture_mode)

    actual_adc_fs = PL_CLK_HZ / (2 * adc_half_period)
    actual_save_fs = actual_adc_fs / decimation

    print("sample_count    =", sample_count)
    print("capture_mode    =", capture_mode)
    print("adc_half_period =", adc_half_period)
    print("PL_CLK_HZ       =", PL_CLK_HZ)
    print("actual_adc_fs   =", actual_adc_fs)
    print("decimation      =", decimation)
    print("actual_save_fs  =", actual_save_fs)

    return sample_count, actual_adc_fs, actual_save_fs

def run_capture(sample_count, capture_mode, timeout_s=5.0):
    buf = allocate(shape=(BUFFER_WORDS,), dtype=np.int32)
    buf[:] = -12345
    buf.flush()

    writer.write(WRITER_BUFFER, int(buf.physical_address) & 0xFFFFFFFF)
    writer.write(WRITER_COUNT, sample_count)
    writer.write(WRITER_CAPTURE_MODE, capture_mode)

    if capture_mode == 0:
        # HLS fake，不需要启动 capture_core
        writer.write(WRITER_AP_CTRL, 0x01)
    else:
        # stream 模式：writer 先启动，再 start capture
        ctrl.write(CTRL, 0x01)      # enable
        writer.write(WRITER_AP_CTRL, 0x01)
        ctrl.write(CTRL, 0x03)      # enable + start pulse
        ctrl.write(CTRL, 0x01)      # release start

    t0 = time()
    while True:
        ctrl_status = ctrl.read(STATUS)
        writer_ctrl = writer.read(WRITER_AP_CTRL)

        writer_ap_done = (writer_ctrl & 0x2) != 0
        writer_ap_idle = (writer_ctrl & 0x4) != 0
        writer_done = writer_ap_done or writer_ap_idle

        ctrl_done = (ctrl_status & (1 << 1)) != 0

        if capture_mode == 0:
            done = writer_done
        else:
            done = writer_done and ctrl_done

        if done:
            break

        if time() - t0 > timeout_s:
            print("TIMEOUT DEBUG:")
            read_status()
            raise TimeoutError("capture timeout")

    buf.invalidate()

    ch0 = np.array(buf[0:sample_count], dtype=np.int32)
    ch1 = np.array(buf[MAX_SAMPLE_N:MAX_SAMPLE_N + sample_count], dtype=np.int32)

    return buf, ch0, ch1

def plot_capture(ch0, ch1, fs=None, title="capture", max_points=None):
    if max_points is not None:
        ch0 = ch0[:max_points]
        ch1 = ch1[:max_points]

    n = len(ch0)
    if fs is None:
        x = np.arange(n)
        xlabel = "sample index"
    else:
        x = np.arange(n) / fs
        xlabel = "time / s"

    plt.figure(figsize=(12, 4))
    plt.plot(x, ch0, label="CH0")
    plt.plot(x, ch1, label="CH1")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("raw code")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("CH0 first 8:", ch0[:8])
    print("CH1 first 8:", ch1[:8])
    print("CH0 mean/vpp:", float(np.mean(ch0)), int(np.max(ch0) - np.min(ch0)))
    print("CH1 mean/vpp:", float(np.mean(ch1)), int(np.max(ch1) - np.min(ch1)))

def save_csv(ch0, ch1, filename="ad9226_capture_data.csv"):
    import pandas as pd
    df = pd.DataFrame({
        "index": np.arange(len(ch0)),
        "ch0_raw": ch0,
        "ch1_raw": ch1,
    })
    df.to_csv(filename, index=False)
    print("Saved:", filename)
    return df

## 2. 基础状态检查

如果你前面已经测过，可以快速跑一下确认 overlay 正常。

In [ ]:
print("VERSION =", hex(ctrl.read(VERSION)))
read_status()

## 3. 可选：Quick Mode 0/2 自检

如果刚刚重新上传 bit/hwh，建议先跑。  
如果你已经确认过 mode 0 和 mode 2，可以跳过。

In [ ]:
# Quick mode 0: HLS fake -> DDR
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=1024,
    adc_half_period=25,
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=0
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=0)
read_status()
plot_capture(ch0, ch1, title="quick mode 0: HLS fake -> DDR", max_points=512)

In [ ]:
# Quick mode 2: RTL fake stream -> FIFO -> HLS -> DDR
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=1024,
    adc_half_period=25,
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=2
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=2)
read_status()
plot_capture(ch0, ch1, fs=actual_save_fs, title="quick mode 2: RTL fake stream -> FIFO -> HLS -> DDR", max_points=512)

## 4. ADC clock 手动查看

接逻辑分析仪：

```text
GND -> PYNQ / AD9226 GND
CH0 -> adc_a_clk
CH1 -> adc_b_clk
```

当前 `PL_CLK_HZ=31.25MHz`，所以：

| adc_half_period | 预期 ADC clock |
|---:|---:|
| 125 | 125 kHz |
| 50 | 312.5 kHz |
| 25 | 625 kHz |
| 12 | 1.302 MHz |
| 6 | 2.604 MHz |
| 2 | 7.8125 MHz |
| 1 | 15.625 MHz |

In [ ]:
# 手动改这里：125 / 50 / 25 / 12 / 6 / 2 / 1
hp = 25

sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=1024,
    adc_half_period=hp,
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=1
)

# 只 enable，不 start writer，不 start capture
ctrl.write(CTRL, 0x01)

sleep(1.0)
read_status()

print("Expected ADC clock Hz:", actual_adc_fs)
print("Expected ADC clock MHz:", actual_adc_fs / 1e6)

## 5. 真实 AD9226 低速采样

先建议：

```text
adc_half_period = 25   -> 约 625 kHz
sample_count = 1024
```

如果没有接输入信号，数据可能固定或乱跳。接入信号后再判断波形质量。

In [ ]:
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=1024,
    adc_half_period=25,   # 当前实际约 625 kHz
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=1
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=1, timeout_s=5.0)
read_status()
plot_capture(ch0, ch1, fs=actual_save_fs, title="mode 1: real AD9226, Fs≈625kHz")

## 6. 扫 sample_delay

如果真实波形不稳定，先扫 `sample_delay`。  
建议先在低速下扫，比如 `adc_half_period=25`。

In [ ]:
for sd in [0, 1, 2, 3]:
    print("\nTesting sample_delay =", sd)

    sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
        sample_count=1024,
        adc_half_period=25,  # 约 625 kHz
        sample_delay=sd,
        decimation=1,
        channel_mask=0b11,
        capture_mode=1
    )

    try:
        buf, ch0, ch1 = run_capture(sample_count, capture_mode=1, timeout_s=5.0)
        read_status()
        plot_capture(ch0, ch1, fs=actual_save_fs, title=f"real ADC sample_delay={sd}", max_points=512)
    except Exception as e:
        print("FAILED:", e)
        read_status()

## 7. 逐步升频真实采样

当前 PL 时钟是 31.25MHz，最高 `adc_half_period=1` 也只是 15.625MSPS。  
先按 625kHz → 1.3MHz → 2.6MHz → 7.8MHz → 15.6MHz 走。

In [ ]:
for hp in [25, 12, 6, 2, 1]:
    print("\nTesting adc_half_period =", hp)

    sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
        sample_count=4096,
        adc_half_period=hp,
        sample_delay=1,
        decimation=1,
        channel_mask=0b11,
        capture_mode=1
    )

    try:
        buf, ch0, ch1 = run_capture(sample_count, capture_mode=1, timeout_s=5.0)
        read_status()
        plot_capture(ch0, ch1, fs=actual_save_fs, title=f"real ADC hp={hp}, Fs={actual_adc_fs/1e6:.3f} MHz", max_points=1024)
    except Exception as e:
        print("FAILED at hp =", hp, e)
        read_status()
        break

## 8. 大点数测试

真实低速稳定后再测试大点数。

In [ ]:
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=16384,
    adc_half_period=25,   # 约 625 kHz
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=1
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=1, timeout_s=8.0)
read_status()
plot_capture(ch0, ch1, fs=actual_save_fs, title="real AD9226 16384 samples", max_points=2000)

## 9. 保存当前数据 CSV

In [ ]:
df = save_csv(ch0, ch1, "ad9226_capture_data.csv")
df.head()